In [2]:
import numpy as np
import pandas as pd
import geopandas as gpd
from scipy.spatial.distance import pdist, squareform
from scipy.sparse import csr_matrix, lil_matrix
from pyproj import Transformer
from sklearn.preprocessing import StandardScaler
import pyreadr 
from pathlib import Path
import os

os.chdir(Path.cwd().parent)


In [5]:
import numpy as np
import pandas as pd
import pyreadr
from pathlib import Path
from tqdm import tqdm

# ================================================================
# CONFIG
# ================================================================
BASE_DIR = Path(r"D:\77\Research\temp\snow")
period = 52

no_nbs = np.array([
    57,170,236,269,343,685,946,947,989,
    1037,1084,1090,1109,1118,1127,1176,1203
]) - 1

# ================================================================
# LOAD DATA (NON-ISOLATED ONLY)
# ================================================================
snow = pyreadr.read_r("snow_cleaned_full.Rda")
snow = list(snow.values())[0].reset_index(drop=True)

y_full = snow.iloc[:, 2:].to_numpy()
coords_full = snow.iloc[:, :2].to_numpy()

y = np.delete(y_full, no_nbs, axis=0)
coords = np.delete(coords_full, no_nbs, axis=0)

S, T = y.shape

# ================================================================
# GLOBAL TIME VARIABLES (MATCH MCMC)
# ================================================================
t_raw = np.arange(1, T + 1)
t_trend_full = (t_raw - t_raw.mean()) / t_raw.std(ddof=0)

# ================================================================
# COVARIATES
# ================================================================
cov_bym = np.column_stack([
    np.ones(T), np.ones(T),
    np.cos(2*np.pi*t_raw/period), np.cos(2*np.pi*t_raw/period),
    np.sin(2*np.pi*t_raw/period), np.sin(2*np.pi*t_raw/period),
    t_trend_full, t_trend_full
])   # (T, 8)

cov_iid = np.column_stack([
    np.ones(T),
    np.cos(2*np.pi*t_raw/period),
    np.sin(2*np.pi*t_raw/period),
    t_trend_full
])   # (T, 4)

# ================================================================
# FACTOR COMPONENTS (MATCH run_bym_factor)
# ================================================================
# latitude (global scale)
lat_raw = coords[:, 1]
lats = (lat_raw - lat_raw.mean()) / lat_raw.std(ddof=1)

# elevation (global scale)
curr_elev = pd.read_csv("curr_elev.csv").iloc[:, 3].to_numpy()
elev = (curr_elev - curr_elev.mean()) / curr_elev.std(ddof=1)

# temperature (global scale)
snow_temp = pyreadr.read_r("snow_temp_full.Rda")
snow_temp = list(snow_temp.values())[0].reset_index(drop=True)
temp_full = snow_temp.iloc[:, 2:].to_numpy()
temp = np.delete(temp_full, no_nbs, axis=0)

temp_scaled = (temp - temp.mean()) / temp.std(ddof=0)

# ================================================================
# LOAD POSTERIOR SAMPLES
# ================================================================
iid01  = np.load(BASE_DIR / "ind01_noIso.npz")["all_theta"]
iid10  = np.load(BASE_DIR / "ind10_noIso.npz")["all_theta"]

bym01  = np.load(BASE_DIR / "bym01_noIso_final.npz")["all_theta"]
bym10  = np.load(BASE_DIR / "bym10_noIso_final.npz")["all_theta"]

bymf01 = np.load(BASE_DIR / "bym_factor_01_noiso.npz")["all_theta"]
bymf10 = np.load(BASE_DIR / "bym_factor_10_noiso.npz")["all_theta"]

M = iid01.shape[1]

# ================================================================
# JOINT LOG-LIKELIHOOD
# ================================================================
def loglik_joint(y, theta01, theta10, cov01, cov10,
                 lats=None, elev=None, temp_scaled=None, t_trend=None):

    S, T = y.shape
    ll = 0.0
    K = cov01.shape[1]
    use_factor = lats is not None

    for t in range(1, T):
        eta01 = np.zeros(S)
        eta10 = np.zeros(S)

        for k in range(K):
            eta01 += cov01[t, k] * theta01[k*S:(k+1)*S]
            eta10 += cov10[t, k] * theta10[k*S:(k+1)*S]

        if use_factor:
            g01 = theta01[K*S:K*S+3]
            g10 = theta10[K*S:K*S+3]

            eta01 += t_trend[t] * (
                g01[0]*lats + g01[1]*elev + g01[2]*temp_scaled[:, t]
            )
            eta10 += t_trend[t] * (
                g10[0]*lats + g10[1]*elev + g10[2]*temp_scaled[:, t]
            )

        p01 = 1 / (1 + np.exp(-eta01))
        p10 = 1 / (1 + np.exp(-eta10))

        prob = np.where(y[:, t-1] == 0, p01, 1 - p10)

        ll += np.sum(
            y[:, t] * np.log(prob + 1e-12)
            + (1 - y[:, t]) * np.log(1 - prob + 1e-12)
        )

    return ll

# ================================================================
# DIC WRAPPER
# ================================================================
def compute_dic(label, theta01, theta10, cov01, cov10, use_factor=False):

    ll = np.zeros(M)

    for m in tqdm(range(M), desc=f"DIC ({label})"):
        ll[m] = loglik_joint(
            y,
            theta01[:, m],
            theta10[:, m],
            cov01, cov10,
            lats if use_factor else None,
            elev if use_factor else None,
            temp_scaled if use_factor else None,
            t_trend_full if use_factor else None
        )

    ll_bar = ll.mean()

    th01_bar = theta01.mean(axis=1)
    th10_bar = theta10.mean(axis=1)

    ll_hat = loglik_joint(
        y, th01_bar, th10_bar, cov01, cov10,
        lats if use_factor else None,
        elev if use_factor else None,
        temp_scaled if use_factor else None,
        t_trend_full if use_factor else None
    )

    D_bar = -2 * ll_bar
    D_hat = -2 * ll_hat
    p_D = D_bar - D_hat

    return {
        "DIC": D_bar + p_D,
        "p_D": p_D,
        "loglik_mean": ll_bar,
        "loglik_at_mean": ll_hat
    }

# ================================================================
# RUN ALL THREE
# ================================================================
out_iid  = compute_dic("IID",        iid01,  iid10,  cov_iid, cov_iid)
out_bym  = compute_dic("BYM",        bym01,  bym10,  cov_bym, cov_bym)
out_bymf = compute_dic("BYM+factor", bymf01, bymf10, cov_bym, cov_bym, use_factor=True)

# ================================================================
# PRINT SUMMARY
# ================================================================
print("\n===== JOINT DIC COMPARISON =====\n")

for name, out in zip(
    ["IID", "BYM", "BYM + factor"],
    [out_iid, out_bym, out_bymf]
):
    print(name)
    for k, v in out.items():
        print(f"  {k}: {v:.3f}")
    print()

print("ΔDIC (BYM − IID):",  out_bym["DIC"]  - out_iid["DIC"])
print("ΔDIC (BYM+ − BYM):", out_bymf["DIC"] - out_bym["DIC"])
print("ΔDIC (BYM+ − IID):", out_bymf["DIC"] - out_iid["DIC"])


DIC (BYM+factor): 100%|██████████| 1000/1000 [08:13<00:00,  2.03it/s]



===== JOINT DIC COMPARISON =====

IID
  DIC: 1306158.385
  p_D: 5871.258
  loglik_mean: -650143.564
  loglik_at_mean: -647207.935

BYM
  DIC: 1301466.096
  p_D: 2714.770
  loglik_mean: -649375.663
  loglik_at_mean: -648018.278

BYM + factor
  DIC: 1293031.458
  p_D: 2721.908
  loglik_mean: -645154.775
  loglik_at_mean: -643793.822

ΔDIC (BYM − IID): -4692.289844811428
ΔDIC (BYM+ − BYM): -8434.63713912759
ΔDIC (BYM+ − IID): -13126.926983939018
